# Этап 1. Работа с пропусками. 

Цель -  понять семантику NA в кажой колонке и орабатывать пропуски корректно 

**Ключевая идея:** NA ≠ NA.

- Группа A — NA = «нет объекта» (категориальные) → `"None"`
- Группа B — NA = «нет объекта» (числовые) → `0`
- Группа C — NA = «данные потеряны» → умная импутация



### Шаг 1.1 Изучение `data_description.txt`

Прошлись по 19 колонкам с пропусками. Выводы:

- Группа A - NA "Нет объекта" 15 колонок, категориальные признаки 

В описании прямо указано, что NA - "нет объекта" 

| Колонка | Значение NA |
|---|---|
| PoolQC | No Pool |
| MiscFeature | No MiscFeature |
| Alley | No alley access |
| Fence | No Fence |
| FireplaceQu | No Fireplace |
| GarageType | No Garage |
| GarageFinish | No Garage |
| GarageQual | No Garage |
| GarageCond | No Garage |
| BsmtQual | No Basement |
| BsmtCond | No Basement |
| BsmtExposure | No Basement |
| BsmtFinType1 | No Basement |
| BsmtFinType2 | No Basement |
| MasVnrType | No Veneer |

Итог: заполнить ячейки "None"


- Группа B - NA "Нет объекта" 2 колонки, числовые значения 

| Колонка | Значение NA |
|---|---|
| GarageYrBlt | Нет гаража → нет года |
| MasVnrArea | Нет облицовки → нет площади |

Итог: заполнить ячейки 0

- Группа C - NA "Нет объекта" 2 колонки, данные потеряны 

 Колонка | Тип |
|---|---|
| LotFrontage | 259 пропусков, данные реально потеряны |
| Electrical | 1 пропуск |

Итог: вычисление медианы по районам для `LotFrontage`, моды для `Electrical`


Изучение данных осуществлялось посредством вывода значений колонок в терминале: grep -A 6 "GarageFinish" data/data_description.txt

In [16]:
import json
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.2f}".format)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

### Шаг 1.2. Загрузка данных и списков колонок

Загружаем оригинальные `train.csv` и `test.csv` (с пропусками) и списки колонок из `columns_info.json`.

In [17]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

print("train: ", train.shape)
print("test: ", test.shape)


train:  (1460, 81)
test:  (1459, 80)


In [18]:
with open("../columns_info.json") as f:
    cols_info = json.load(f)
    
target = cols_info["target"]
id_col = cols_info["id_col"]
pure_num_cols = cols_info["pure_num_cols"]
pseudo_num_cols = cols_info["pseudo_num_cols"]
cat_cols = cols_info["cat_cols"]

print(f"Числовых: {len(pure_num_cols)}")
print(f"Псевдочисловых: {len(pseudo_num_cols)}")
print(f"Категориальных: {len(cat_cols)}")

Числовых: 33
Псевдочисловых: 3
Категориальных: 43


### Шаг 1.3. Пропуски до обработки

подсчет пропусков до обработки

In [19]:
missing_before = train.isna().sum().sort_values(ascending = False)
missing_before = missing_before[missing_before > 0]

print(f"колонок с пропусками: {len(missing_before)}")
print(f"Всего пропусков: {train.isna().sum().sum()}")
print()
print(missing_before)

колонок с пропусками: 19
Всего пропусков: 7829

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageYrBlt       81
GarageCond        81
GarageType        81
GarageFinish      81
GarageQual        81
BsmtFinType2      38
BsmtExposure      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64


### Шаг 1.4. Заполнение группы A — «нет объекта» (категориальные)

Вместо NA вставляем "None"

In [20]:
none_cat_cols = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "MasVnrType",
]
print("Пропусков до: ", train[none_cat_cols].isna().sum().sum())

for col in none_cat_cols:
    train[col] = train[col].fillna("__None__")
    test[col] = test[col].fillna("__None__")
    
print(f"пропусков после: ", train[none_cat_cols].isna().sum().sum())
        

Пропусков до:  7480
пропусков после:  0


In [21]:
print("Всего пропусков в train:", train.isna().sum().sum())
print("Всего пропусков в test :", test.isna().sum().sum())



Всего пропусков в train: 349
Всего пропусков в test : 342


In [22]:
# проверяем, где остались пропуски 

missing_now = train.isna().sum().sort_values(ascending=False)
missing_now = missing_now[missing_now > 0]
print(missing_now)

LotFrontage    259
GarageYrBlt     81
MasVnrArea       8
Electrical       1
dtype: int64


### Шаг 1.5. Заполнение группы B — «нет объекта» (числовые)

`GarageYrBlt` (год постройки гаража) и `MasVnrArea` (площадь облицовки). NA означает, что объекта нет → заполняем `0`.

In [23]:
none_num_cols = ["GarageYrBlt", "MasVnrArea"]

print("Пропусков до: ", train[none_num_cols].isna().sum().sum())

for col in none_num_cols:
    train[col] = train[col].fillna(0)
    test[col]  = test[col].fillna(0)
    
print("Пропусков после: ", train[none_num_cols].isna().sum().sum())

Пропусков до:  89
Пропусков после:  0


### Шаг 1.6. Умная импутация LotFrontage

259 пропусков (17.7%). Данные реально утеряны. Глобальная медиана не подходит: дома в одном районе похожи по ширине фасада. Заполняем медианой **по Neighborhood**.

In [24]:
print("Пропусков до: ", train["LotFrontage"].isna().sum())

for df in [train, test]:
    df["LotFrontage"] = df.groupby("Neighborhood")["LotFrontage"] \
        .transform(lambda x: x.fillna(x.median()))

print("Пропусков после:", train["LotFrontage"].isna().sum())

Пропусков до:  259
Пропусков после: 0


### Шаг 1.7. Заполнение Electrical

1 пропуск в категориальной колонке. Заполняем модой (самое частое значение).

In [25]:
print("До: ", train["Electrical"].isna().sum())
electrical_mode = train["Electrical"].mode().iloc[0]
print(f"Мода Electrical: {electrical_mode}")


for df in [train, test]:
    df["Electrical"] = df["Electrical"].fillna(electrical_mode)
    
print("После: ",  train["Electrical"].isna().sum())

До:  1
Мода Electrical: SBrkr
После:  0


### Шаг 1.8. Заполнение пропусков в test

В test остались 22 пропуска в 15 колонках (их не было в train). Применяем ту же логику: мода из train для категориальных, медиана из train для числовых.

In [26]:

cat_cols_test = ["MSZoning", "Utilities", "Functional", "Exterior1st",
                 "Exterior2nd", "SaleType", "KitchenQual"]

for col in cat_cols_test:
    mode_val = train[col].mode().iloc[0]    # мода по train
    test[col] = test[col].fillna(mode_val)  # заполняем NaN этой модой
    print(f"{col}: заполнено {mode_val}")


num_cols_test = ["BsmtFullBath", "BsmtHalfBath", "BsmtFinSF1",
                 "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
                 "GarageCars", "GarageArea"]

for col in num_cols_test:
    median_val = train[col].median()         # медиана по train
    test[col] = test[col].fillna(median_val) # заполняем NaN этой медианой
    print(f"{col}: заполнено {median_val}")


print()
print("Всего пропусков в test:", test.isna().sum().sum())

MSZoning: заполнено RL
Utilities: заполнено AllPub
Functional: заполнено Typ
Exterior1st: заполнено VinylSd
Exterior2nd: заполнено VinylSd
SaleType: заполнено WD
KitchenQual: заполнено TA
BsmtFullBath: заполнено 0.0
BsmtHalfBath: заполнено 0.0
BsmtFinSF1: заполнено 383.5
BsmtFinSF2: заполнено 0.0
BsmtUnfSF: заполнено 477.5
TotalBsmtSF: заполнено 991.5
GarageCars: заполнено 2.0
GarageArea: заполнено 480.0

Всего пропусков в test: 0


### Шаг 1.9. Сохранение очищенных данных

Сохраняем обработанные DataFrame в **новые** файлы: `train_clean.csv`, `test_clean.csv`. Оригиналы `train.csv` / `test.csv` не трогаем.

In [27]:

train.to_csv("../data/train_clean.csv", index=False)
test.to_csv("../data/test_clean.csv", index=False)
print("Сохранено: train_clean.csv, test_clean.csv")

Сохранено: train_clean.csv, test_clean.csv


In [28]:

import pandas as pd

train_check = pd.read_csv("../data/train_clean.csv")
test_check  = pd.read_csv("../data/test_clean.csv")

print("Файл train_clean.csv — пропусков:", train_check.isna().sum().sum())
print("Файл test_clean.csv  — пропусков:", test_check.isna().sum().sum())
print()
print("Размеры:")
print("  train_clean:", train_check.shape)
print("  test_clean :", test_check.shape)

Файл train_clean.csv — пропусков: 0
Файл test_clean.csv  — пропусков: 0

Размеры:
  train_clean: (1460, 81)
  test_clean : (1459, 80)


In [29]:

import os
print(os.listdir("../data"))

['train_clean.csv', 'test.csv', 'data_description.txt', 'train.csv', 'test_clean.csv']


## Итог по этапу 1

| Группа | Колонок | Что делали | До → После |
|---|---|---|---|
| A — «нет объекта» (кат.) | 15 | Заполнили `"None"` | 7480 → 0 |
| B — «нет объекта» (числ.) | 2 | Заполнили `0` | 89 → 0 |
| C — `LotFrontage` | 1 | Медиана по Neighborhood | 259 → 0 |
| C — `Electrical` | 1 | Мода (SBrkr) | 1 → 0 |
| **Итого** | **19** | | **7829 → 0** |

